### Summary
- **Model type:** Linear programming (continuous decision variables)
- **Objective:** Minimize total hourly generation cost
- **Constraints:** Meet demand exactly and respect each generator's capacity
- **Solver:** HiGHS via Pyomo framework

## Problem

A small grid must meet a demand of **150 MW** using three generators. Choose each generator's capacity to minimize total hourly production cost.


| Generator | Cost (£/MWh) | Max capacity (MW) |
|---|---:|---:|
| Gas A | 45 | 80 |
| Gas B | 55 | 70 |
| Oil | 80 | 50 |



## Mathematical Model

Minimize total hourly cost:

\begin{align}
    \min_{p_1,p_2,p_3}~~ & 45p_1 + 55p_2 + 80p_3 \\
    \text{s.t.}~~ & p_1 + p_2 + p_3 = 150 \\
                  & p_1 \le 80 \\
                  & p_2 \le 70 \\
                  & p_3 \le 50 \\
                  & p_1, p_2, p_3 \ge 0
\end{align}

In [ ]:
# Import libraries
import pandas as pd
import pyomo.environ as pyo
import pyomo.opt as po

In [13]:
# Input data
generators = pd.DataFrame({
    "Generator": ["Gas A", "Gas B", "Oil"],
    "Cost (£/MWh)": [45, 55, 80],
    "Max capacity (MW)": [80, 70, 50],
}).set_index("Generator")

demand = 150

display(generators)
print(f"Demand: {demand} MW")
print(f"Total available capacity: {generators['Max capacity (MW)'].sum()} MW")

,Cost (£/MWh),Max capacity (MW)
Generator,,
Gas A,45,80
Gas B,55,70
Oil,80,50


Demand: 150 MW
Total available capacity: 200 MW


In [ ]:
# Pyomo Model
def solve_dispatch(data, demand, solver_name="appsi_highs", tee=False):
    """
    Solve the energy dispatch and return model
    """
    model = pyo.ConcreteModel()
    model.G = pyo.Set(initialize=list(data.index))

    costs = data["Cost (£/MWh)"].to_dict()
    capacities = data["Max capacity (MW)"].to_dict()

    model.p = pyo.Var(
        model.G,
        domain=pyo.NonNegativeReals,
        bounds=lambda m, g: (0, capacities[g]),
    )

    model.total_cost = pyo.Objective(
        expr=sum(costs[g] * model.p[g] for g in model.G),
        sense=pyo.minimize,
    )

    model.demand_balance = pyo.Constraint(
        expr=sum(model.p[g] for g in model.G) == demand
    )

    solver = pyo.SolverFactory(solver_name)
    if not solver.available(exception_flag=False):
        raise RuntimeError(
            f"Solver '{solver_name}' is unavailable."
        )

    results = solver.solve(model, tee=tee)
    return model, results


model, results = solve_dispatch(generators, demand)

if (results.solver.status == po.SolverStatus.ok and
        results.solver.termination_condition == po.TerminationCondition.optimal):
    dispatch = pd.DataFrame({
        "Cost (£/MWh)": generators["Cost (£/MWh)"],
        "Max capacity (MW)": generators["Max capacity (MW)"],
        "Dispatch (MW)": {g: pyo.value(model.p[g]) for g in model.G},
    })
    dispatch["Cost (£/h)"] = dispatch["Cost (£/MWh)"] * dispatch["Dispatch (MW)"]
    display(dispatch.round(2))
    print(f"Total generation: {sum(pyo.value(model.p[g]) for g in model.G):.1f} MW")
    print(f"Minimum hourly cost: £{pyo.value(model.total_cost):,.2f}/h")
else:
    raise RuntimeError(
        f"Optimization did not reach an optimum: status={results.solver.status}, "
        f"termination={results.solver.termination_condition}"
    )

,Cost (£/MWh),Max capacity (MW),Dispatch (MW),Cost (£/h)
Gas A,45,80,80.0,3600.0
Gas B,55,70,70.0,3850.0
Oil,80,50,0.0,0.0


Total generation: 150.0 MW
Minimum hourly cost: £7,450.00/h


<div style="border-radius:10px; border:#8EACCD solid; padding: 15px; background-color: #bfdbe4; font-size:100%; text-align:left">
<h3 align="left"><font color='#c99527'>💡Insights</font></h3>

<p style="color: #5795db;">The expected dispatch is:</p>

<ul style="color: #5795db;">
  <li>Gas A: <strong>80 MW</strong></li>
  <li>Gas B:<strong> 70 MW</strong></li>
  <li>Oil:<strong> 0 MW</strong></li>
</ul>

<p style="color: #5795db;"> with a minimum hourly cost of <strong>£7,450/h</strong>.</p>

<p style="color: #5795db;">The optimizer uses the lowest-cost generation first, subject to each unit's capacity. For a demand of 150 MW, Gas A and Gas B together can supply the full demand (80 MW + 70 MW), so the oil unit is not needed.</p>

## Experiments

Use the same model-building function to explore how demand and generator costs affect the optimal solution.

In [17]:
def run_experiment(data, demand):
    """
    Solve a scenario and display dispatch, cost or infeasibility
    """
    total_capacity = data["Max capacity (MW)"].sum()
    if demand > total_capacity:
        print(
            f"Infeasible before solving: demand is {demand} MW, "
            f"but total capacity is only {total_capacity} MW."
        )
        return

    scenario_model, scenario_results = solve_dispatch(data, demand)
    optimal = (
        scenario_results.solver.status == po.SolverStatus.ok and
        scenario_results.solver.termination_condition == po.TerminationCondition.optimal
    )
    if not optimal:
        print("No optimal solution:", scenario_results.solver.termination_condition)
        return

    values = {g: pyo.value(scenario_model.p[g]) for g in scenario_model.G}
    output = data.copy()
    output["Dispatch (MW)"] = pd.Series(values)
    output["Cost (£/h)"] = output["Cost (£/MWh)"] * output["Dispatch (MW)"]
    display(output.round(2))
    print(f"Demand met: {sum(values.values()):.1f} MW")
    print(f"Total hourly cost: £{pyo.value(scenario_model.total_cost):,.2f}/h")

In [18]:
# Experiment 1: Increase demand to 190 MW.
print("Experiment 1 — Demand = 190 MW")
run_experiment(generators, 190)

Experiment 1 — Demand = 190 MW


,Cost (£/MWh),Max capacity (MW),Dispatch (MW),Cost (£/h)
Generator,,,,
Gas A,45,80,80.0,3600.0
Gas B,55,70,70.0,3850.0
Oil,80,50,40.0,3200.0


Demand met: 190.0 MW
Total hourly cost: £10,650.00/h


<div style="border-radius:10px; border:#8EACCD solid; padding: 15px; background-color: #bfdbe4; font-size:100%; text-align:left">

<p style="color: #5795db;">  Experiment 1 interpretation: Gas A and Gas B reach their combined capacity of 150 MW, so the oil generator supplies the remaining 40 MW.</p>

In [ ]:
# Experiment 2: Increase Gas A's cost to £65/MWh.
generators_cost_change = generators.copy()
generators_cost_change.loc["Gas A", "Cost (£/MWh)"] = 65

print("Experiment 2 — Gas A cost = £65/MWh; demand = 150 MW")
run_experiment(generators_cost_change, 150)

Experiment 2 — Gas A cost = £65/MWh; demand = 150 MW


,Cost (£/MWh),Max capacity (MW),Dispatch (MW),Cost (£/h)
Generator,,,,
Gas A,65,80,80.0,5200.0
Gas B,55,70,70.0,3850.0
Oil,80,50,0.0,0.0


Demand met: 150.0 MW
Total hourly cost: £9,050.00/h


<div style="border-radius:10px; border:#8EACCD solid; padding: 15px; background-color: #bfdbe4; font-size:100%; text-align:left">
<p style="color: #5795db;">  Experiment 2 interpretation: The dispatch remains Gas A = 80 MW and Gas B = 70 MW. Even after Gas A's cost rises to £65/MWh, Gas A and Gas B are still cheaper than oil, and their combined capacity exactly meets the 150 MW demand.</p>

In [20]:
# Experiment 3: Demand above total capacity is infeasible.
print("Experiment 3 — Demand = 210 MW")
run_experiment(generators, 210)

Experiment 3 — Demand = 210 MW
Infeasible before solving: demand is 210 MW, but total capacity is only 200 MW.


<div style="border-radius:10px; border:#8EACCD solid; padding: 15px; background-color: #bfdbe4; font-size:100%; text-align:left">
<p style="color: #5795db;">  Experiment 3 interpretation: Total capacity is 80 + 70 + 50 = 200 MW. A demand of 210 MW cannot be met, so the problem is infeasible. A demand of exactly 200 MW would still be feasible, with all three generators at maximum output.</p>